In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import matplotlib.pyplot as plt

from selenium.common.exceptions import TimeoutException
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from scipy.stats import gaussian_kde
import numpy as np

import asyncio
from requests_html import AsyncHTMLSession
import nest_asyncio
# Function to extract match data from a single page
def extract_matches(soup):
    data = []
    for sublist in soup.find_all("div", class_="results-sublist"):
        for result_con in sublist.find_all("div", class_="result-con"):
            a_reset = result_con.find("a", class_="a-reset")
            if a_reset:
                result = a_reset.find("div", class_="result")

                # Extract team names, score, and match link
                team1 = result.find_all("td", class_="team-cell")[0].text.strip()
                score = result.find("td", class_="result-score").text.strip()
                team2 = result.find_all("td", class_="team-cell")[1].text.strip()
                match_link = "https://www.hltv.org" + a_reset["href"]

                # Create a dictionary for each match with relevant data
                match_data = {
                    "Team 1": team1,
                    "Team 2": team2,
                    "Score": score,
                    "Match Link": match_link,
                }
                data.append(match_data)

    return data
# Set up the WebDriver (without specifying the driver path)
driver = webdriver.Firefox()

# Navigate to the first page
url = "https://www.hltv.org/results"
driver.get(url)

# Wait for the cookie banner to appear (up to 10 seconds) and click it
try:
    wait = WebDriverWait(driver, 10)
    cookie_banner = wait.until(EC.presence_of_element_located((By.XPATH, "//*[@id='CybotCookiebotDialogBodyButtonDecline']")))
    cookie_banner.click()
except TimeoutException:
    print("Cookie banner not found. Continuing with scraping...")

# Scrape match history for a specified number of pages
num_pages = 1  # Change this value to the desired number of pages
match_history = []

for page in range(1, num_pages + 1):
    offset = (page - 1) * 100
    url = f"https://www.hltv.org/results?offset={offset}"

    driver.get(url)

    # Get the HTML content
    html_content = driver.page_source

    # Parse the HTML content with BeautifulSoup
    soup = BeautifulSoup(html_content, "html.parser")

    match_history.extend(extract_matches(soup))

# Close the WebDriver
driver.quit()

# Create a pandas DataFrame with the scraped data
df = pd.DataFrame(match_history)

# Save the DataFrame to an Excel file
df.to_excel("match_history.xlsx", index=False)

# Look at winrates by adding a column of who won
# Step 1: Create a new column 'Outcome'
def get_winner(score, team1, team2):
    s1, s2 = map(int, score.split(" - "))
    if s1 > s2:
        return team1
    if s1 < s2:
        return team2
    else:
        return "Draw"

df["Outcome"] = df[["Score", "Team 1", "Team 2"]].apply(lambda row: get_winner(row["Score"], row["Team 1"], row["Team 2"]), axis=1)

# Step 2: Create a new DataFrame 'bo1s' for rows with a score superior to 3 to account for matches that are either BO1 or BO5
def is_score_superior_to_3(score):
    s1, s2 = map(int, score.split(" - "))
    return s1 > 3 or s2 > 3

bo1s = df[df["Score"].apply(is_score_superior_to_3)].copy()
df = df[~df["Score"].apply(is_score_superior_to_3)]


In [ ]:
df

In [ ]:
import os

chromium_executable_path = r"C:\Users\Coco\OneDrive - Erasmus University Rotterdam\Coding\Betting hltv\Chrome webdriver\chrome-win\chrome.exe"

async def extract_map_names_requests_html(match_link):
    async with AsyncHTMLSession(executable_path=chromium_executable_path) as session:
        response = await session.get(match_link)
        await response.html.arender()

       


In [ ]:
import nest_asyncio
nest_asyncio.apply()

import asyncio
import pandas as pd
from requests_html import AsyncHTMLSession

async def extract_map_names_requests_html(match_link):
    session = AsyncHTMLSession()
    response = await session.get(match_link)
    await response.html.arender()

    # Now, use the rendered HTML with requests_html's built-in selectors or BeautifulSoup
    map_names = []

    # Step 1: Find the <div class="flexbox-column"> element
    flexbox_column = response.html.find("div.flexbox-column", first=True)

    if flexbox_column:
        # Step 2: Iterate through the child elements with <div class="mapholder">
        for mapholder in flexbox_column.find_all("div.mapholder"):
            # Step 3: For each mapholder, find the child element with <div class="played">
            played = mapholder.find("div.played", first=True)

            if played:
                # Step 4: Inside the played element, find the child element with <div class="map-name-holder">
                map_name_holder = played.find("div.map-name-holder", first=True)

                if map_name_holder:
                    # Step 5: Extract the map name from the map-name-holder element
                    map_name_element = map_name_holder.find("div.mapname", first=True)

                    if map_name_element:
                        map_name = map_name_element.text
                        map_names.append(map_name)

    await session.close()
    return map_names

async def process_links(df):
    tasks = [extract_map_names_requests_html(row["Match Link"]) for index, row in df.iterrows()]
    results = await asyncio.gather(*tasks)

    for index, result in enumerate(results):
        df.at[index, "Map Data"] = result

# Assuming 'df' is your existing DataFrame with match links
asyncio.run(process_links(df))


In [ ]:
df

### Version with playwright ###

In [ ]:
import pandas as pd
from playwright.sync_api import sync_playwright

def extract_map_names_playwright(match_link):
    with sync_playwright() as p:
        browser = p.chromium.launch()
        context = browser.new_context()
        page = context.new_page()

        page.goto(match_link)

        # Now, use the rendered HTML with Playwright's selectors or BeautifulSoup
        map_names = []

        # Step 1: Find the <div class="flexbox-column"> element
        flexbox_column = page.query_selector("div.flexbox-column")

        if flexbox_column:
            # Step 2: Iterate through the child elements with <div class="mapholder">
            mapholders = flexbox_column.query_selector_all("div.mapholder")
            for mapholder in mapholders:
                # Step 3: For each mapholder, find the child element with <div class="played">
                played = mapholder.query_selector("div.played")

                if played:
                    # Step 4: Inside the played element, find the child element with <div class="map-name-holder">
                    map_name_holder = played.query_selector("div.map-name-holder")

                    if map_name_holder:
                        # Step 5: Extract the map name from the map-name-holder element
                        map_name_element = map_name_holder.query_selector("div.mapname")

                        if map_name_element:
                            map_name = map_name_element.inner_text()
                            map_names.append(map_name)

        browser.close()
        return map_names

def process_links(df):
    results = [extract_map_names_playwright(row["Match Link"]) for index, row in df.iterrows()]

    for index, result in enumerate(results):
        df.at[index, "Map Data"] = result

# Assuming 'df' is your existing DataFrame with match links
process_links(df)


In [ ]:
import pandas as pd
import asyncio
import playwright

async def extract_map_names_playwright(match_link):
    async with playwright.async_playwright() as p:
        browser = await p.chromium.launch()
        context = await browser.new_context()
        page = await context.new_page()

        await page.goto(match_link)

        # Now, use the rendered HTML with Playwright's selectors or BeautifulSoup
        map_names = []

        # Step 1: Find the <div class="flexbox-column"> element
        flexbox_column = await page.query_selector("div.flexbox-column")

        if flexbox_column:
            # Step 2: Iterate through the child elements with <div class="mapholder">
            mapholders = await flexbox_column.query_selector_all("div.mapholder")
            for mapholder in mapholders:
                # Step 3: For each mapholder, find the child element with <div class="played">
                played = await mapholder.query_selector("div.played")

                if played:
                    # Step 4: Inside the played element, find the child element with <div class="map-name-holder">
                    map_name_holder = await played.query_selector("div.map-name-holder")

                    if map_name_holder:
                        # Step 5: Extract the map name from the map-name-holder element
                        map_name_element = await map_name_holder.query_selector("div.mapname")

                        if map_name_element:
                            map_name = await map_name_element.inner_text()
                            map_names.append(map_name)

        await browser.close()
        return map_names

async def process_links(df):
    tasks = []
    for index, row in df.iterrows():
        tasks.append(extract_map_names_playwright(row["Match Link"]))

    results = await asyncio.gather(*tasks)

    for index, result in enumerate(results):
        df.at[index, "Map Data"] = result

async def main():
    # Assuming 'df' is your existing DataFrame with match links
    await process_links(df)
    print(df)

# Create a sample DataFrame with match links
df = pd.DataFrame({
    "Match Link": [
        "https://www.hltv.org/matches/2347574/astralis-vs-complexity-blast-premier-fall-2020",
        "https://www.hltv.org/matches/2347575/heroic-vs-vitality-blast-premier-fall-2020",
        "https://www.hltv.org/matches/2347576/nip-vs-faze-blast-premier-fall-2020"
    ]
})

# Run the main function
asyncio.run(main())
